# DKA Training — Google Colab (Cloud GPU)
**Dynamic Kernel Attention — DKA-Small on CIFAR-10**

Make sure you have a GPU runtime: Runtime > Change runtime type > T4/A100/L4


In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Dynamic-Kernel-Attention.git'
REPO_DIR = '/content/Dynamic-Kernel-Attention'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install deps not in Colab by default
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'datasets', 'transformers', 'wandb', 'seaborn', 'pyyaml'], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')


In [ ]:
# ============================================================
# CONFIGURATION — DKA-Small (full run)
# ============================================================
import yaml

# Override config for cloud GPU
config = {
    "experiment": {"name": "dka-small-cifar10-colab", "seed": 42},
    "model": {
        "name": "DKA-Small",
        "config_name": "small",
        "type": "image",
        "d_model": 256,
        "num_heads": 8,
        "num_layers": 8,
        "rank": 4,
        "kernel_sizes": [3, 3, 3, 3, 5, 5, 5, 5],  # 2D side lengths: 3x3=9 and 5x5=25 neighbors
    },
    "data": {
        "dataset": "cifar10",
        "data_dir": "/content/data",
        "image_size": 32,
        "patch_size": 4,
        "num_classes": 10,
        "in_channels": 3,
    },
    "training": {
        "batch_size": 1024,
        "epochs": 300,
        "num_workers": 2,
        "pin_memory": True,
    },
    "optimizer": {
        "main_lr": 3e-4,
        "kernel_gen_lr": 3e-5,
        "alpha_lr": 1e-3,
        "weight_decay": 0.05,
    },
    "scheduler": {"warmup_epochs": 10, "min_lr": 1e-5},
    "regularization": {
        "dropout": 0.1,
        "drop_path": 0.1,
        "label_smoothing": 0.1,
    },
    "augmentation": {
        "mixup_alpha": 0.8,
        "cutmix_alpha": 1.0,
        "mixup_prob": 0.5,
    },
    "ema": {"enabled": True, "decay": 0.9999},
    "diversity_loss": {"lambda_div": 0.1, "tau": 0.5, "num_pairs": 64},
    "grad_clip": {"global_max_norm": 1.0, "kernel_gen_max_norm": 0.5},
    "logging": {
        "use_wandb": False,  # Set True and add API key if you want W&B
        "wandb_project": "dka-cifar10",
        "log_every_n_steps": 50,
    },
    "checkpointing": {
        "save_dir": "/content/checkpoints",
        "save_every_n_epochs": 50,
        "save_best": True,
        "best_metric": "val_accuracy",
    },
    "baselines": {"enabled": False, "model_name": "vit"},
}

cfg = config

# --- Automatic LR scaling for batch size ---
REFERENCE_BATCH_SIZES = {
    "cifar10": 256, "tinyimagenet": 128, "agnews": 64, "wikitext2": 64,
}
dataset_name = cfg["data"]["dataset"]
batch_size = cfg["training"]["batch_size"]
ref_bs = REFERENCE_BATCH_SIZES.get(dataset_name, batch_size)
if batch_size != ref_bs:
    lr_scale = math.sqrt(batch_size / ref_bs)
    cfg["optimizer"]["main_lr"] *= lr_scale
    cfg["optimizer"]["kernel_gen_lr"] *= lr_scale
    cfg["optimizer"]["alpha_lr"] *= lr_scale
    print(f"LR scaled by {lr_scale:.3f}x for batch_size={batch_size} (ref={ref_bs})")

print(f"Config loaded: DKA-Small, CIFAR-10, 300 epochs, batch {batch_size}")

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
from dka.data.cifar10 import get_cifar10_loaders, MixupCutMix

batch_size = cfg['training']['batch_size']
num_workers = cfg['training']['num_workers']
data_dir = cfg['data']['data_dir']

train_loader, val_loader, mixup_cutmix_fn = get_cifar10_loaders(
    data_dir=data_dir,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=cfg['training']['pin_memory'],
)
num_classes = 10
task = 'classification'

# Disable mixup/cutmix if alphas are 0
aug_cfg = cfg.get('augmentation', {})
mixup_a = aug_cfg.get('mixup_alpha', 0)
cutmix_a = aug_cfg.get('cutmix_alpha', 0)
if mixup_a == 0 and cutmix_a == 0:
    mixup_cutmix_fn = None
    print('Mixup/CutMix: DISABLED')
else:
    print(f'Mixup/CutMix: ENABLED (mixup={mixup_a}, cutmix={cutmix_a})')

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
print(f'Batch size: {batch_size}')


In [ ]:
# ============================================================
# CREATE MODEL
# ============================================================
from dka.models.dka_model import DKAImageModel

device = DEVICE

model = DKAImageModel.from_config(
    cfg['model']['config_name'],
    img_size=cfg['data']['image_size'],
    patch_size=cfg['data']['patch_size'],
    num_classes=num_classes,
    dropout=cfg['regularization']['dropout'],
    drop_path_rate=cfg['regularization']['drop_path'],
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {cfg["model"]["name"]}')
print(f'Total params: {total_params:,} ({total_params/1e6:.2f}M)')
print(f'Trainable params: {trainable_params:,}')
print(f'Device: {device}')


In [ ]:
# ============================================================
# OPTIMIZER, SCHEDULER, LOSS
# ============================================================
from dka.training.optimizer import build_optimizer
from dka.training.scheduler import build_scheduler
from dka.training.ema import EMA

optimizer = build_optimizer(
    model,
    main_lr=cfg['optimizer']['main_lr'],
    kernel_gen_lr=cfg['optimizer']['kernel_gen_lr'],
    alpha_lr=cfg['optimizer']['alpha_lr'],
    weight_decay=cfg['optimizer']['weight_decay'],
)

steps_per_epoch = len(train_loader)
total_steps = cfg['training']['epochs'] * steps_per_epoch
scheduler = build_scheduler(
    optimizer,
    total_steps=total_steps,
    warmup_epochs=cfg['scheduler']['warmup_epochs'],
    steps_per_epoch=steps_per_epoch,
    eta_min=cfg['scheduler']['min_lr'],
)

label_smoothing = cfg['regularization']['label_smoothing']
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

from dka.training.losses import KernelDiversityLoss
div_cfg = cfg['diversity_loss']
div_loss_fn = KernelDiversityLoss(num_pairs=div_cfg['num_pairs'], tau=div_cfg['tau'])
lambda_div = div_cfg['lambda_div']

ema = None
if cfg['ema']['enabled']:
    ema = EMA(model, decay=cfg['ema']['decay'])

scaler = torch.amp.GradScaler('cuda')

print(f'Optimizer: AdamW ({len(optimizer.param_groups)} param groups)')
print(f'Scheduler: Warmup ({cfg["scheduler"]["warmup_epochs"]} epochs) + Cosine -> {cfg["scheduler"]["min_lr"]}')
print(f'Total steps: {total_steps:,}')
print(f'EMA: {"enabled" if ema else "disabled"}')
print(f'AMP: enabled (CUDA)')


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
import time
from pathlib import Path
from collections import defaultdict
from dka.training.optimizer import clip_grad_norms

num_epochs = cfg["training"]["epochs"]
save_dir = Path(cfg["checkpointing"]["save_dir"])
save_dir.mkdir(parents=True, exist_ok=True)
save_every = cfg["checkpointing"]["save_every_n_epochs"]
log_every = cfg["logging"]["log_every_n_steps"]
grad_clip_cfg = cfg["grad_clip"]
div_loss_every = 10  # Compute diversity loss every N steps

best_val_acc = 0.0
history = defaultdict(list)
alpha_history = []

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    model.train()
    
    running_loss = 0.0
    running_ce = 0.0
    running_div = 0.0
    correct = 0
    total = 0
    
    for step, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Mixup/CutMix
        mixed_labels = labels
        if mixup_cutmix_fn is not None:
            images, mixed_labels = mixup_cutmix_fn(images, labels)
        
        optimizer.zero_grad()
        
        # Forward with AMP
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(images)
            ce_loss = criterion(logits, mixed_labels)
            
            # Diversity loss (every N steps)
            div_loss = torch.tensor(0.0, device=device)
            if step % div_loss_every == 0:
                kernel_list = []
                for block in model.blocks:
                    kernels = block.dka.get_last_kernels()
                    if kernels:
                        kernel_list.extend(kernels.values())
                if kernel_list:
                    div_loss = div_loss_fn(kernel_list)
            
            loss = ce_loss + lambda_div * div_loss
        
        # Backward
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            clip_grad_norms(model,
                          max_norm_global=grad_clip_cfg["global_max_norm"],
                          max_norm_kernel_gen=grad_clip_cfg["kernel_gen_max_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            clip_grad_norms(model,
                          max_norm_global=grad_clip_cfg["global_max_norm"],
                          max_norm_kernel_gen=grad_clip_cfg["kernel_gen_max_norm"])
            optimizer.step()
        
        scheduler.step()
        if ema is not None:
            ema.update()
        
        running_loss += loss.item()
        running_ce += ce_loss.item()
        running_div += div_loss.item()
        
        # Accuracy (use original labels, not mixed)
        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        # Logging
        global_step = (epoch - 1) * len(train_loader) + step
        if step % log_every == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"  Step {step}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} (CE: {ce_loss.item():.4f}, Div: {div_loss.item():.4f}) | "
                  f"LR: {lr:.2e}")
    
    # Epoch stats
    n = len(train_loader)
    train_loss = running_loss / n
    train_ce = running_ce / n
    train_div = running_div / n
    train_acc = correct / total
    
    # --- Validation ---
    model.eval()
    if ema is not None:
        ema.apply()
    
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=scaler is not None):
                logits = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    if ema is not None:
        ema.restore()
    
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    elapsed = time.time() - t0
    
    # Track alpha values
    alphas = {}
    for i, block in enumerate(model.blocks):
        block_alphas = block.dka.get_last_alphas()
        if block_alphas:
            for h, a in block_alphas.items():
                alphas[f"L{i}_H{h}"] = a.item()
    alpha_history.append(alphas)
    
    # History
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch}/{num_epochs} [{elapsed:.1f}s] | "
          f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")
    
    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "config": cfg,
        }, save_dir / "best_model.pt")
        print(f"  -> New best val_acc: {val_acc:.4f} (saved)")
    
    # Periodic save
    if epoch % save_every == 0:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "config": cfg,
        }, save_dir / f"checkpoint_epoch{epoch}.pt")

print(f"
Training complete\! Best val accuracy: {best_val_acc:.4f}")


In [ ]:
# ============================================================
# TRAINING CURVES
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"], label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Alpha trajectories
if alpha_history:
    keys = sorted(alpha_history[0].keys())
    for key in keys:
        vals = [ah.get(key, 0) for ah in alpha_history]
        axes[2].plot(vals, label=key, alpha=0.7)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Alpha")
    axes[2].set_title("Alpha Trajectories")
    axes[2].legend(fontsize=6, ncol=2)
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /content/training_curves.png")


In [ ]:
# ============================================================
# DOWNLOAD BEST CHECKPOINT
# ============================================================
from google.colab import files

checkpoint_path = "/content/checkpoints/best_model.pt"
if os.path.exists(checkpoint_path):
    files.download(checkpoint_path)
    print(f"Downloading {checkpoint_path}")
else:
    print("No checkpoint found. Did training complete?")
